# ARCH Technologies Internship - Task 3- Optical Character Recognition (OCR)
 Use Python to build a basic OCR system that can extract text from images
 containing printed or handwritten text. Work with libraries like Tesseract and
 OpenCV to preprocess the images, apply OCR, and display the extracted text. Test the system on various sample images and see the results. Dataset to use: **IAM Handwritten Forms Dataset on Kaggle**

# 1. Install Required Libraries


In [ ]:
# Install Tesseract OCR engine and Python libraries
!sudo apt update
!sudo apt install tesseract-ocr -y
!pip install pytesseract opencv-python pillow matplotlib numpy

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,816 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,594 kB]
Get:14 https://r

# 2. Upload IAM Handwritten Dataset ZIP File

In [ ]:
from google.colab import files

# Upload the ZIP file containing the IAM dataset (e.g., 'iam_handwriting.zip')
# Click "Choose Files" and select your downloaded ZIP file
uploaded = files.upload()

# 3. Extract the Uploaded ZIP File

In [ ]:
import zipfile
import os

# Get the name of the uploaded ZIP file
zip_filename = list(uploaded.keys())[0]

# Extract all contents to a folder named 'dataset'
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print(f"✅ Dataset extracted successfully to 'dataset/' folder.")

# 4. Define Image Preprocessing Function Using OpenCV

In [ ]:
import cv2
import numpy as np

def preprocess_image(image_path):
    """
    Preprocesses an image to improve OCR accuracy.
    Steps:
      1. Load image in grayscale
      2. Denoise to reduce noise
      3. Apply Otsu's thresholding for binarization
    """
    # Load image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found at path: {image_path}")

    # Denoise the image (reduces small artifacts)
    denoised = cv2.fastNlMeansDenoising(img, None, h=10, templateWindowSize=7, searchWindowSize=21)

    # Apply binary threshold using Otsu's method (automatically finds best threshold)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return binary

# 5. Set Up Tesseract OCR and Define Text Extraction Function

In [ ]:
import pytesseract

# Set Tesseract command path (default in Colab is usually correct)
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

def extract_text_from_image(image_path):
    """
    Extracts text from a preprocessed image using Tesseract OCR.
    Uses PSM 6 (assume a single uniform block of text) and English language.
    """
    processed_img = preprocess_image(image_path)

    # Configure Tesseract: OEM 3 (default LSTM), PSM 6 (block of text), language = English
    custom_config = r'--oem 3 --psm 6 -l eng'

    # Perform OCR
    extracted_text = pytesseract.image_to_string(processed_img, config=custom_config)

    return extracted_text.strip()

# 6. Locate and Display Sample Images from the Dataset


In [ ]:
import os
import matplotlib.pyplot as plt

# Define the folder containing form images (adjust if your structure differs)
forms_dir = "dataset/forms"

# Check if the folder exists
if not os.path.exists(forms_dir):
    # Try common alternative paths
    possible_dirs = ["dataset/data/forms", "dataset/Forms", "dataset/images"]
    for d in possible_dirs:
        if os.path.exists(d):
            forms_dir = d
            break
    else:
        raise FileNotFoundError("Could not find 'forms' directory. Please check your dataset structure.")

# Get list of image files (PNG/JPG)
image_files = [f for f in os.listdir(forms_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Select first 3 images for testing
test_images = image_files[:3] if len(image_files) >= 3 else image_files

print(f"📁 Found {len(image_files)} images. Testing on: {test_images}")

# 7. Run OCR on Sample Images and Display Results


In [ ]:
# Process each selected image
for img_name in test_images:
    img_path = os.path.join(forms_dir, img_name)

    # Load original image for display
    original_img = cv2.imread(img_path)
    rgb_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

    # Display original image
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(rgb_img)
    plt.title(f"Original Image: {img_name}")
    plt.axis("off")

    # Show preprocessed image
    processed = preprocess_image(img_path)
    plt.subplot(1, 2, 2)
    plt.imshow(processed, cmap='gray')
    plt.title("Preprocessed (Binarized)")
    plt.axis("off")
    plt.show()

    # Extract and print text
    try:
        text = extract_text_from_image(img_path)
        print(f"\n OCR Result for '{img_name}':\n")
        print(text if text else "[No text detected]")
    except Exception as e:
        print(f"\n Error processing '{img_name}': {str(e)}")

    print("\n" + "="*70 + "\n")